# **01 — Brasileirão 2020–2025: coleta de dados**

Este notebook cobre as seis temporadas completas do Brasileirão Série A entre 2020 e 2025.

Para cada partida, o fluxo extrai 22 titulares, preserva cinco reservas com nascimento conhecido e sorteia um deles.

O dataset público (transfermarkt-datasets, baixado em `data/raw/transfermarkt/`) fornece as escalações de 2025. As edições de 2020–2024 são completadas diretamente pelas súmulas do Transfermarkt e armazenadas em cache.

A análise do paradoxo do aniversário fica no notebook seguinte, **02 — Paradoxo do aniversário**.


In [ ]:
import random
import re
import threading
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from math import prod, sqrt
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)

BASE_URL = "https://www.transfermarkt.com"
COMPETITION_ID = "BRA1"
MAX_WORKERS = 3
REQUEST_START_INTERVAL = 0.5
MAX_RETRIES = 3
REFRESH_HTML = False

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

TM_DIR = PROJECT_DIR / "data" / "raw" / "transfermarkt"
CACHE_DIR = TM_DIR / "brasileirao_2020_2025"
CALENDAR_DIR = CACHE_DIR / "calendars"
REPORT_DIR = CACHE_DIR / "match_reports"
PROFILE_DIR = TM_DIR / "player_profiles"
OUTPUT_DIR = PROJECT_DIR / "data" / "brasileirao_2020_2025"
ANALYSIS_DIR = PROJECT_DIR / "data" / "analysis"
for directory in (CALENDAR_DIR, REPORT_DIR, PROFILE_DIR, OUTPUT_DIR, ANALYSIS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = "https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data"


def download_dataset(filename):
    destination = TM_DIR / filename
    if destination.exists() and destination.stat().st_size > 0:
        return destination
    temporary = destination.with_suffix(destination.suffix + ".part")
    with requests.get(f"{DATA_BASE_URL}/{filename}", stream=True, timeout=300) as response:
        response.raise_for_status()
        with temporary.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
    temporary.replace(destination)
    return destination


LINEUPS_DATASET_PATH = download_dataset("game_lineups.csv.gz")
PLAYERS_DATASET_PATH = download_dataset("players.csv.gz")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9,pt-BR;q=0.8",
}
_rate_lock = threading.Lock()
_last_request_start = 0.0

## **1. Mapeamento das temporadas**

O identificador técnico usado pelo Transfermarkt fica um ano atrás do nome do Brasileirão.

In [ ]:
SEASONS = pd.DataFrame({
    "brasileirao_season": [2020, 2021, 2022, 2023, 2024, 2025],
    "transfermarkt_season_id": [2019, 2020, 2021, 2022, 2023, 2024],
})
display(SEASONS)

## **2. Download concorrente e controlado**

No máximo três páginas são processadas simultaneamente. O início das requisições é espaçado e cada HTML válido é salvo imediatamente.

In [ ]:
def valid_html(html):
    return len(html) > 30_000 and "Access Denied" not in html


def wait_for_request_slot():
    global _last_request_start
    with _rate_lock:
        elapsed = time.monotonic() - _last_request_start
        if elapsed < REQUEST_START_INTERVAL:
            time.sleep(REQUEST_START_INTERVAL - elapsed)
        _last_request_start = time.monotonic()


def fetch_cached_page(url, cache_path):
    if cache_path.exists() and not REFRESH_HTML:
        html = cache_path.read_text(encoding="utf-8")
        if valid_html(html):
            return html, "cache"
    for attempt in range(1, MAX_RETRIES + 1):
        wait_for_request_slot()
        try:
            response = requests.get(url, headers=HEADERS, timeout=45)
            if response.status_code == 200 and valid_html(response.text):
                temporary = cache_path.with_name(
                    f"{cache_path.name}.{threading.get_ident()}.tmp"
                )
                temporary.write_text(response.text, encoding="utf-8")
                temporary.replace(cache_path)
                return response.text, "download"
            if response.status_code != 429 and response.status_code < 500:
                break
        except requests.RequestException as error:
            if attempt == MAX_RETRIES:
                warnings.warn(f"Falha em {url}: {error}")
        if attempt < MAX_RETRIES:
            time.sleep(2 ** attempt)
    return None, "error"

## **3. Calendários e 2.280 partidas**

In [ ]:
def calendar_url(season_id):
    return f"{BASE_URL}/x/gesamtspielplan/wettbewerb/{COMPETITION_ID}/saison_id/{season_id}"


def parse_club_id(link):
    match = re.search(r"/verein/(\d+)", link.get("href", "")) if link else None
    return match.group(1) if match else None


def parse_calendar(html, brasileirao_season, season_id):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for heading in soup.select("div.content-box-headline"):
        round_match = re.match(r"^\s*(\d+)\s*\.", heading.get_text(" ", strip=True))
        if round_match is None:
            continue
        box = heading.find_parent("div", class_="box")
        current_date = None
        for tr in box.select("tbody tr") if box else []:
            date_link = tr.select_one('a[href*="/datum/"]')
            if date_link:
                date_match = re.search(r"/datum/(\d{4}-\d{2}-\d{2})", date_link.get("href", ""))
                if date_match:
                    current_date = date_match.group(1)
            home = tr.select_one('td.text-right.hauptlink a[href*="/verein/"]')
            away = tr.select_one('td.no-border-links.hauptlink a[href*="/verein/"]')
            report = tr.select_one('a[href*="/spielbericht/"]')
            if home is None or away is None or report is None:
                continue
            game_match = re.search(r"/spielbericht/(\d+)", report.get("href", ""))
            score = re.search(r"(\d+)\s*:\s*(\d+)", report.get_text(" ", strip=True))
            rows.append({
                "game_id": game_match.group(1) if game_match else None,
                "brasileirao_season": brasileirao_season,
                "transfermarkt_season_id": season_id,
                "round": int(round_match.group(1)),
                "date": current_date,
                "home_club_id": parse_club_id(home),
                "away_club_id": parse_club_id(away),
                "home_team": home.get("title") or home.get_text(" ", strip=True),
                "away_team": away.get("title") or away.get_text(" ", strip=True),
                "home_goals": int(score.group(1)) if score else None,
                "away_goals": int(score.group(2)) if score else None,
                "match_report_url": urljoin(BASE_URL, report.get("href", "")),
            })
    return rows

In [ ]:
all_matches = []
for season in SEASONS.to_dict("records"):
    season_id = season["transfermarkt_season_id"]
    html, source = fetch_cached_page(
        calendar_url(season_id), CALENDAR_DIR / f"{season_id}.html"
    )
    all_matches.extend(parse_calendar(html, season["brasileirao_season"], season_id))

matches_df = pd.DataFrame(all_matches).drop_duplicates("game_id")
matches_df["game_id"] = matches_df["game_id"].astype("string")
matches_df["home_club_id"] = matches_df["home_club_id"].astype("string")
matches_df["away_club_id"] = matches_df["away_club_id"].astype("string")
matches_df["date"] = pd.to_datetime(matches_df["date"], errors="coerce").dt.date
matches_df = matches_df.sort_values(["brasileirao_season", "round", "date", "game_id"])

assert len(matches_df) == 2280
assert matches_df.groupby("brasileirao_season").size().eq(380).all()
assert matches_df["game_id"].notna().all()
matches_df.to_csv(OUTPUT_DIR / "brasileirao_2020_2025_matches.csv", index=False)
display(matches_df.groupby("brasileirao_season").agg(
    matches=("game_id", "size"), first_date=("date", "min"), last_date=("date", "max")
))

## **4. Escalações do dataset e súmulas históricas**

In [ ]:
def parse_player_link(link):
    match = re.search(r"/([^/]+)/profil/spieler/(\d+)", link.get("href", ""))
    if match is None:
        return None
    return {
        "player_id": match.group(2),
        "player_name": link.get("title") or link.get_text(" ", strip=True),
        "player_code": match.group(1),
        "player_url": urljoin(BASE_URL, link.get("href", "")),
    }


def parse_direct_lineup(html, game_id):
    soup = BeautifulSoup(html, "html.parser")
    heading = next((node for node in soup.select("h2.content-box-headline")
                    if "line-up" in node.get_text(" ", strip=True).lower()), None)
    lineup_box = heading.find_parent("div", class_="box") if heading else None
    team_boxes = []
    for candidate in lineup_box.find_all("div") if lineup_box else []:
        if ("large-6" in candidate.get("class", [])
                and candidate.find("div", class_="aufstellung-unterueberschrift-mannschaft",
                                   recursive=False)):
            team_boxes.append(candidate)
    rows = []
    for side, box in zip(("home", "away"), team_boxes[:2]):
        club_link = box.select_one('.aufstellung-unterueberschrift-mannschaft a[href*="/verein/"]')
        club_id = parse_club_id(club_link)
        groups = (
            ("starting_lineup", box.select('div.formation-player-container a[href*="/profil/spieler/"]')),
            ("substitutes", box.select('table.ersatzbank a[href*="/profil/spieler/"]')),
        )
        for lineup_type, links in groups:
            for link in links:
                player = parse_player_link(link)
                if player:
                    rows.append({
                        "game_id": str(game_id), "club_id": club_id, "team_side": side,
                        "type": lineup_type, "lineup_source": "match_report", **player,
                    })
    return rows


def fetch_and_parse_match(match):
    game_id = str(match["game_id"])
    html, source = fetch_cached_page(
        match["match_report_url"], REPORT_DIR / f"{game_id}.html"
    )
    return parse_direct_lineup(html, game_id) if html else []

In [ ]:
target_game_ids = set(matches_df["game_id"])
dataset_parts = []
for chunk in tqdm(pd.read_csv(
    LINEUPS_DATASET_PATH,
    dtype={"game_id": "string", "player_id": "string", "club_id": "string", "number": "string"},
    chunksize=500_000,
), desc="dataset de escalações"):
    selected = chunk.loc[chunk["game_id"].isin(target_game_ids)].copy()
    if not selected.empty:
        dataset_parts.append(selected)

dataset_lineups_df = pd.concat(dataset_parts, ignore_index=True)
dataset_lineups_df["lineup_source"] = "dataset"
side_lookup = matches_df.set_index("game_id")[["home_club_id", "away_club_id"]]
dataset_lineups_df = dataset_lineups_df.join(side_lookup, on="game_id")
dataset_lineups_df["team_side"] = "unknown"
dataset_lineups_df.loc[
    dataset_lineups_df["club_id"] == dataset_lineups_df["home_club_id"], "team_side"
] = "home"
dataset_lineups_df.loc[
    dataset_lineups_df["club_id"] == dataset_lineups_df["away_club_id"], "team_side"
] = "away"
dataset_lineups_df["player_code"] = pd.NA
dataset_lineups_df["player_url"] = pd.NA

covered_ids = set(dataset_lineups_df["game_id"])
direct_matches = matches_df.loc[~matches_df["game_id"].isin(covered_ids)]
direct_rows = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(fetch_and_parse_match, match): match["game_id"]
        for match in direct_matches.to_dict("records")
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="súmulas históricas"):
        try:
            direct_rows.extend(future.result())
        except Exception as error:
            warnings.warn(f"Partida {futures[future]}: {error}")

direct_lineups_df = pd.DataFrame(direct_rows)
columns = [
    "game_id", "player_id", "club_id", "player_name", "type", "team_side",
    "lineup_source", "player_code", "player_url",
]
frames = [dataset_lineups_df.reindex(columns=columns)]
if not direct_lineups_df.empty:
    frames.append(direct_lineups_df.reindex(columns=columns))
lineups_df = pd.concat(frames, ignore_index=True)
lineups_df = lineups_df.drop_duplicates(["game_id", "player_id", "type"])

print(f"Partidas via dataset: {len(covered_ids)}")
print(f"Partidas tentadas diretamente: {len(direct_matches)}")
print(f"Partidas com escalação: {lineups_df.game_id.nunique()}")

## **5. Jogadores únicos e datas de nascimento**

In [ ]:
def parse_birth_date(html):
    soup = BeautifulSoup(html, "html.parser")
    element = soup.select_one('[itemprop="birthDate"]')
    if element:
        match = re.search(r"(\d{2}/\d{2}/\d{4})", element.get_text(" ", strip=True))
        if match:
            return pd.to_datetime(match.group(1), format="%d/%m/%Y")
    return pd.NaT


def fetch_player_birth(player):
    player_id = str(player["player_id"])
    player_code = player["player_code"] if pd.notna(player["player_code"]) else "x"
    unavailable = PROFILE_DIR / f"player-{player_id}.unavailable"
    if unavailable.exists() and not REFRESH_HTML:
        return player_id, pd.NaT
    html, source = fetch_cached_page(
        f"{BASE_URL}/{player_code}/profil/spieler/{player_id}",
        PROFILE_DIR / f"player-{player_id}.html",
    )
    if html is None:
        unavailable.write_text(player_id, encoding="utf-8")
        return player_id, pd.NaT
    unavailable.unlink(missing_ok=True)
    return player_id, parse_birth_date(html)

In [ ]:
catalog_df = pd.read_csv(
    PLAYERS_DATASET_PATH,
    dtype={"player_id": "string"},
    usecols=["player_id", "name", "player_code", "date_of_birth", "url"],
).rename(columns={
    "name": "catalog_name", "player_code": "catalog_player_code",
    "url": "catalog_player_url",
})
lineups_df = lineups_df.merge(catalog_df, on="player_id", how="left")
lineups_df["player_name"] = lineups_df["player_name"].fillna(lineups_df["catalog_name"])
lineups_df["player_code"] = lineups_df["player_code"].fillna(lineups_df["catalog_player_code"])
lineups_df["player_url"] = lineups_df["player_url"].fillna(lineups_df["catalog_player_url"])
lineups_df["birth_date"] = pd.to_datetime(lineups_df["date_of_birth"], errors="coerce")

unique_players_df = lineups_df[
    ["player_id", "player_name", "player_code", "player_url", "birth_date"]
].drop_duplicates("player_id").reset_index(drop=True)
missing_players = unique_players_df.loc[unique_players_df["birth_date"].isna()].to_dict("records")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(fetch_player_birth, player) for player in missing_players]
    for future in tqdm(as_completed(futures), total=len(futures), desc="perfis únicos"):
        player_id, birth_date = future.result()
        if pd.notna(birth_date):
            unique_players_df.loc[
                unique_players_df["player_id"] == player_id, "birth_date"
            ] = birth_date

birth_lookup = unique_players_df.set_index("player_id")["birth_date"]
lineups_df["birth_date"] = lineups_df["player_id"].map(birth_lookup)
unique_players_df.to_csv(OUTPUT_DIR / "brasileirao_2020_2025_unique_players.csv", index=False)
lineups_df.to_csv(OUTPUT_DIR / "brasileirao_2020_2025_all_lineups.csv", index=False)

print(f"Jogadores únicos: {len(unique_players_df)}")
print(f"Com nascimento: {unique_players_df.birth_date.notna().sum()}")
print(f"Sem nascimento: {unique_players_df.birth_date.isna().sum()}")

## **6. Cinco reservas válidos e seleção dos 23**

In [ ]:
bench_rows = []
for game_id, group in lineups_df.loc[lineups_df["type"] == "substitutes"].groupby("game_id"):
    candidates = group.sort_values("player_id").to_dict("records")
    random.Random(str(game_id)).shuffle(candidates)
    valid = [candidate for candidate in candidates if pd.notna(candidate["birth_date"])]
    for rank, candidate in enumerate(valid[:5], start=1):
        bench_rows.append({**candidate, "reserve_candidate_rank": rank})

bench_candidates_df = pd.DataFrame(bench_rows)
selected_reserves_df = bench_candidates_df.loc[
    bench_candidates_df["reserve_candidate_rank"] == 1
].copy()
selected_reserves_df["role"] = "reserva_sorteado"
starters_df = lineups_df.loc[lineups_df["type"] == "starting_lineup"].copy()
starters_df["role"] = "titular"
starters_df["reserve_candidate_rank"] = pd.NA

selected_players_df = pd.concat([starters_df, selected_reserves_df], ignore_index=True)
selected_players_df = selected_players_df.merge(
    matches_df[[
        "game_id", "brasileirao_season", "round", "date", "home_team", "away_team",
    ]], on="game_id", how="left",
)
bench_candidates_df.to_csv(OUTPUT_DIR / "brasileirao_2020_2025_bench_candidates.csv", index=False)
selected_players_df.to_csv(OUTPUT_DIR / "brasileirao_2020_2025_selected_23_players.csv", index=False)

## **7. Cobertura da extração**

In [ ]:
lineup_counts = lineups_df.groupby(["game_id", "type"]).size().unstack(fill_value=0)
known_counts = (
    lineups_df.assign(known=lineups_df["birth_date"].notna())
    .groupby(["game_id", "type"])["known"].sum().unstack(fill_value=0)
)
candidate_counts = bench_candidates_df.groupby("game_id").size().rename("valid_reserve_candidates")
selected_counts = selected_players_df.groupby(["game_id", "role"]).size().unstack(fill_value=0)

coverage_df = matches_df.copy()
coverage_df = coverage_df.join(lineup_counts.add_suffix("_count"), on="game_id")
coverage_df = coverage_df.join(known_counts.add_suffix("_known_birth"), on="game_id")
coverage_df = coverage_df.join(candidate_counts, on="game_id")
coverage_df = coverage_df.join(selected_counts.add_suffix("_selected"), on="game_id").fillna(0)
coverage_df["status"] = "complete"
coverage_df.loc[coverage_df["starting_lineup_count"] == 0, "status"] = "lineup_unavailable"
coverage_df.loc[
    (coverage_df["starting_lineup_count"] > 0) & (coverage_df["starting_lineup_count"] != 22),
    "status",
] = "incomplete_starters"
coverage_df.loc[
    (coverage_df["starting_lineup_count"] == 22)
    & (coverage_df["starting_lineup_known_birth"] != 22),
    "status",
] = "missing_starter_birthdate"
coverage_df.loc[
    (coverage_df["starting_lineup_count"] == 22)
    & (coverage_df["valid_reserve_candidates"] < 5),
    "status",
] = "fewer_than_5_valid_reserves"
coverage_df.to_csv(OUTPUT_DIR / "brasileirao_2020_2025_extraction_coverage.csv", index=False)

display(coverage_df.groupby(["brasileirao_season", "status"]).size().unstack(fill_value=0))

## **Arquivos gerados**

- `brasileirao_2020_2025_matches.csv`: as 2.280 partidas das seis temporadas;
- `brasileirao_2020_2025_all_lineups.csv`: titulares e bancos completos, com data de nascimento;
- `brasileirao_2020_2025_unique_players.csv`: uma linha por jogador;
- `brasileirao_2020_2025_bench_candidates.csv`: cinco reservas válidos por partida;
- `brasileirao_2020_2025_selected_23_players.csv`: 22 titulares + 1 reserva sorteado;
- `brasileirao_2020_2025_extraction_coverage.csv`: auditoria por partida.

Continue no notebook **02 — Paradoxo do aniversário** para a análise estatística.
